In [2]:
%pip install yfinance


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Yahoo Finance provides historical market data for:

Stocks
ETFs
Commodities
Forex
Indices
Cryptocurrencies

In [3]:
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path

Commodity Tickers
GC=F: Gold Futures
SI=F: Silver futures

Future: Imagine an agreement today:
"I will buy Gold next month for ₹7000."
That agreement itself is called a futures contract.

Since futures are actively traded every day, they provide a good representation of commodity prices.

In [4]:
COMMODITIES = {
    "Gold": "GC=F",
    "Silver": "SI=F",
    "Crude_Oil": "CL=F",
    "Natural_Gas": "NG=F",
    "Copper": "HG=F"
}

In [5]:
START_DATE = "2015-01-01"
END_DATE = None  # Downloads data up to today

In [6]:
RAW_DATA_PATH = Path("../data/raw")
RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)

In [8]:

for commodity, ticker in COMMODITIES.items():

    print(f"Downloading {commodity}...")

    df = yf.download(
        ticker,
        start=START_DATE,
        auto_adjust=False,
        progress=False
    )

    # Flatten MultiIndex columns
    if hasattr(df.columns, "nlevels") and df.columns.nlevels > 1:
        df.columns = df.columns.get_level_values(0)

    # Reset index so Date becomes a normal column
    df.reset_index(inplace=True)

    # Save
    df.to_csv(RAW_DATA_PATH / f"{commodity.lower()}.csv", index=False)

print("Download Complete!")













Download Complete!


In [28]:
gold=pd.read_csv("../data/raw/gold.csv")
silver=pd.read_csv("../data/raw/silver.csv")
copper=pd.read_csv("../data/raw/copper.csv")
crude_oil=pd.read_csv("../data/raw/crude_oil.csv")
natural_gas=pd.read_csv("../data/raw/natural_gas.csv")

In [10]:
gold.shape

(2895, 7)

In [11]:
gold.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2895 entries, 0 to 2894
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       2895 non-null   object 
 1   Adj Close  2895 non-null   float64
 2   Close      2895 non-null   float64
 3   High       2895 non-null   float64
 4   Low        2895 non-null   float64
 5   Open       2895 non-null   float64
 6   Volume     2895 non-null   int64  
dtypes: float64(5), int64(1), object(1)
memory usage: 158.4+ KB


In [20]:
gold["Date"]=pd.to_datetime(gold["Date"])

In [13]:
gold.isnull().sum()

Date         0
Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64

In [14]:
gold.duplicated().sum()

np.int64(0)

In [15]:
print(gold["Date"].min())
print(gold["Date"].max())

2015-01-02 00:00:00
2026-07-10 00:00:00


In [16]:
gold.describe()

,Date,Adj Close,Close,High,Low,Open,Volume
count,2895,2895.000000,2895.000000,2895.000000,2895.000000,2895.000000,2895.000000
mean,2020-10-05 13:10:52.849740800,1901.027010,1901.027010,1911.468671,1890.343904,1900.922833,5065.161658
min,2015-01-02 00:00:00,1050.800049,1050.800049,1062.000000,1046.199951,1053.699951,0.000000
25%,2017-11-16 12:00:00,1275.800049,1275.800049,1281.149963,1272.000000,1276.649963,53.000000
50%,2020-10-06 00:00:00,1732.000000,1732.000000,1741.699951,1722.800049,1732.400024,211.000000
75%,2023-08-22 12:00:00,1977.900024,1977.900024,1983.950012,1968.400024,1976.750000,652.000000
max,2026-07-10 00:00:00,5318.399902,5318.399902,5586.200195,5301.600098,5415.700195,386334.000000
std,NaN,886.239123,886.239123,895.124922,877.396139,886.894275,27758.237093


In [21]:
gold.loc[gold["Close"].idxmax()]

Date         2026-01-29 00:00:00
Adj Close            5318.399902
Close                5318.399902
High                 5586.200195
Low                       5097.5
Open                 5415.700195
Volume                     23709
Name: 2783, dtype: object

In [22]:
gold[gold["Volume"] == 0].head()

,Date,Adj Close,Close,High,Low,Open,Volume
728,2017-11-24,1286.699951,1286.699951,1286.699951,1286.699951,1286.699951,0
799,2018-03-12,1319.400024,1319.400024,1319.400024,1319.400024,1319.400024,0
802,2018-03-15,1316.800049,1316.800049,1316.800049,1316.800049,1316.800049,0
807,2018-03-22,1326.599976,1326.599976,1326.599976,1326.599976,1326.599976,0
837,2018-05-04,1312.699951,1312.699951,1312.699951,1312.699951,1312.699951,0


In [17]:
(gold["High"] >= gold["Low"]).all()

np.True_

In [19]:
gold["Volume"].describe()

count      2895.000000
mean       5065.161658
std       27758.237093
min           0.000000
25%          53.000000
50%         211.000000
75%         652.000000
max      386334.000000
Name: Volume, dtype: float64

In [26]:
import sys
from pathlib import Path

sys.path.append(str(Path("../utils").resolve()))

from validation import validate_dataset

In [29]:
gold_summary = validate_dataset(gold, "Gold")
silver_summary = validate_dataset(silver, "Silver")
oil_summary = validate_dataset(crude_oil, "Crude Oil")
gas_summary = validate_dataset(natural_gas, "Natural Gas")
copper_summary = validate_dataset(copper, "Copper")

DATA VALIDATION REPORT : Gold
Shape                : (2895, 7)
Date Range           : 2015-01-02 → 2026-07-10
Missing Values       : 0
Duplicate Rows       : 0
High >= Low          : True
Zero Volume Days     : 37
Minimum Close Price  : 1050.80
Maximum Close Price  : 5318.40
Average Close Price  : 1901.03

Data Types
Date          object
Adj Close    float64
Close        float64
High         float64
Low          float64
Open         float64
Volume         int64
dtype: object

Missing Values by Column
Date         0
Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64

Summary Statistics
         Adj Close        Close         High          Low         Open  \
count  2895.000000  2895.000000  2895.000000  2895.000000  2895.000000   
mean   1901.027010  1901.027010  1911.468671  1890.343904  1900.922833   
std     886.239123   886.239123   895.124922   877.396139   886.894275   
min    1050.800049  1050.800049  1062.000000  1046.199951  1

In [31]:
silver["Date"]=pd.to_datetime(gold["Date"])
natural_gas["Date"]=pd.to_datetime(gold["Date"])
crude_oil["Date"]=pd.to_datetime(gold["Date"])
copper["Date"]=pd.to_datetime(gold["Date"])

In [32]:
summaries = []

summaries.append(validate_dataset(gold, "Gold"))
summaries.append(validate_dataset(silver, "Silver"))
summaries.append(validate_dataset(crude_oil, "Crude Oil"))
summaries.append(validate_dataset(natural_gas, "Natural Gas"))
summaries.append(validate_dataset(copper, "Copper"))

quality_report = pd.DataFrame(summaries)
quality_report

DATA VALIDATION REPORT : Gold
Shape                : (2895, 7)
Date Range           : 2015-01-02 00:00:00 → 2026-07-10 00:00:00
Missing Values       : 0
Duplicate Rows       : 0
High >= Low          : True
Zero Volume Days     : 37
Minimum Close Price  : 1050.80
Maximum Close Price  : 5318.40
Average Close Price  : 1901.03

Data Types
Date         datetime64[ns]
Adj Close           float64
Close               float64
High                float64
Low                 float64
Open                float64
Volume                int64
dtype: object

Missing Values by Column
Date         0
Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64

Summary Statistics
                                Date    Adj Close        Close         High  \
count                           2895  2895.000000  2895.000000  2895.000000   
mean   2020-10-05 13:10:52.849740800  1901.027010  1901.027010  1911.468671   
min              2015-01-02 00:00:00  1050.800049  1

,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Start Date,End Date,High >= Low,Zero Volume Days,Maximum Close,Minimum Close,Average Close
0,Gold,2895,7,0,0,2015-01-02,2026-07-10,True,37,5318.399902,1050.800049,1901.03
1,Silver,2895,7,0,0,2015-01-02,2026-07-10,True,188,115.080002,11.735000,24.40
2,Crude Oil,2896,7,1,0,2015-01-02,2026-07-10,True,7,123.699997,-37.630001,63.14
3,Natural Gas,2897,7,2,0,2015-01-02,2026-07-10,True,7,9.680000,1.482000,3.17
4,Copper,2896,7,1,0,2015-01-02,2026-07-10,True,7,6.649500,1.939500,3.49
